<a href="https://colab.research.google.com/github/paulinepiccio/judging-by-the-cover/blob/main/notebooks/prizes01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

auth.authenticate_user()

In [3]:
PROJECT_ID = "judging-by-the-cover"
client = bigquery.Client(project=PROJECT_ID)

datasets = list(client.list_datasets())
print(f"{PROJECT_ID}")
print(f"{len(datasets)}")
for ds in datasets:
    print(f"{ds.dataset_id}")

judging-by-the-cover
0


In [4]:
DATASET_ID = "literary_prizes"
dataset_ref = client.dataset(DATASET_ID)

try:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "EU"
    dataset.description = "Cross-cultural analysis of major literary prizes 1901-2025"
    dataset = client.create_dataset(dataset, timeout=30)
    print(f"'{DATASET_ID}','{PROJECT_ID}'")
except Exception as e:
    print(f"Error : {e}")

'literary_prizes','judging-by-the-cover'


In [6]:
import pandas as pd
import requests

url = "https://fr.wikipedia.org/wiki/Prix_Goncourt#Liste_des_laur%C3%A9ats"

# Add a User-Agent header to mimic a web browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    # Fetch the HTML content using requests with headers
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

    # Use pandas to read HTML tables from the content
    tables = pd.read_html(response.text)
    print(f"{len(tables)}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")


3


/tmp/ipykernel_3924/2400765927.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [7]:
for i, t in enumerate(tables):
    print(f"\n{'='*60}")
    print(f"--- Tableau {i} (shape: {t.shape}) ---")
    print(f"Colonnes : {list(t.columns)}")
    print(t.head(3))


--- Tableau 0 (shape: (10, 2)) ---
Colonnes : [0, 1]
                    0                   1
0       Prix Goncourt       Prix Goncourt
1                 NaN                 NaN
2  Prix Goncourt 2025  Prix Goncourt 2025

--- Tableau 1 (shape: (123, 6)) ---
Colonnes : ['Année', 'Unnamed: 1', 'Auteur', 'Ouvrage', 'Éditeur  (nombre de prix)', 'Notes']
   Année  Unnamed: 1            Auteur        Ouvrage  \
0   1903         NaN  John-Antoine Nau  Force ennemie   
1   1904         NaN       Léon Frapié  La Maternelle   
2   1905         NaN    Claude Farrère  Les Civilisés   

  Éditeur  (nombre de prix)                                              Notes  
0                  La Plume  Franco-américain d'expression française, premi...  
1              Albin Michel                                                NaN  
2           Paul Ollendorff                                                NaN  

--- Tableau 2 (shape: (3, 2)) ---
Colonnes : ['v\xa0· mPrix Goncourt', 'v\xa0· mPrix Goncourt

In [8]:
df_goncourt = tables[1].copy()

print(df_goncourt.shape)
print(df_goncourt.head(10))
print(df_goncourt.tail(10))
print(df_goncourt.dtypes)
print(df_goncourt.isnull().sum())

(123, 6)
   Année  Unnamed: 1                     Auteur  \
0   1903         NaN           John-Antoine Nau   
1   1904         NaN                Léon Frapié   
2   1905         NaN             Claude Farrère   
3   1906         NaN     Jérôme et Jean Tharaud   
4   1907         NaN              Émile Moselly   
5   1908         NaN       Francis de Miomandre   
6   1909         NaN         Marius-Ary Leblond   
7   1910         NaN              Louis Pergaud   
8   1911         NaN  Alphonse de Châteaubriant   
9   1912         NaN             André Savignon   

                                             Ouvrage  \
0                                      Force ennemie   
1                                      La Maternelle   
2                                      Les Civilisés   
3                       Dingley, l'illustre écrivain   
4  Terres lorraines et  Jean des Brebis ou le Liv...   
5                              Écrit sur de l'eau...   
6                                    

In [9]:
df = tables[1].copy()
df = df.drop(columns=['Unnamed: 1'])
df = df.rename(columns={
    'Année': 'year',
    'Auteur': 'author',
    'Ouvrage': 'book_title',
    'Éditeur  (nombre de prix)': 'publisher_raw',
    'Notes': 'notes'
})

df['publisher'] = df['publisher_raw'].str.replace(r'\s*\(\d+\)\s*$', '', regex=True).str.strip()
df['publisher_cumulative_count'] = df['publisher_raw'].str.extract(r'\((\d+)\)$').astype('Int64')
df = df.drop(columns=['publisher_raw'])

df['prize'] = 'Goncourt'
df['country'] = 'France'

df = df[['prize', 'country', 'year', 'author', 'book_title', 'publisher', 'publisher_cumulative_count', 'notes']]
print(df.head())
print(df.tail())
print(df.info())

      prize country  year                  author  \
0  Goncourt  France  1903        John-Antoine Nau   
1  Goncourt  France  1904             Léon Frapié   
2  Goncourt  France  1905          Claude Farrère   
3  Goncourt  France  1906  Jérôme et Jean Tharaud   
4  Goncourt  France  1907           Émile Moselly   

                                          book_title         publisher  \
0                                      Force ennemie          La Plume   
1                                      La Maternelle      Albin Michel   
2                                      Les Civilisés   Paul Ollendorff   
3                       Dingley, l'illustre écrivain  Édouard Pelletan   
4  Terres lorraines et  Jean des Brebis ou le Liv...              Plon   

   publisher_cumulative_count  \
0                        <NA>   
1                        <NA>   
2                        <NA>   
3                        <NA>   
4                        <NA>   

                                     

In [12]:
df.to_csv('goncourt_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df)} rows")

Saved 123 rows


In [11]:
import requests

url_renaudot = "https://fr.wikipedia.org/wiki/Prix_Renaudot#Prix_Renaudot"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url_renaudot, headers=headers)
    response.raise_for_status()
    tables_renaudot = pd.read_html(response.text)
    print(f"Tables found: {len(tables_renaudot)}")

    for i, t in enumerate(tables_renaudot[:5]):
        print(f"Table {i} - shape {t.shape}")
        print(f"Columns: {list(t.columns)}")
        print(t.head(2))
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")


Tables found: 3
Table 0 - shape (11, 2)
Columns: [0, 1]
                                     0                                    1
0                        Prix Renaudot                        Prix Renaudot
1  Prix Renaudot affiché chez Drouant.  Prix Renaudot affiché chez Drouant.
Table 1 - shape (100, 6)
Columns: ['Année', 'Unnamed: 1', 'Auteur', 'Ouvrage', 'Éditeur (xe fois)', 'Notes']
   Année  Unnamed: 1           Auteur  \
0   1926         NaN     Armand Lunel   
1   1927         NaN  Bernard Nabonne   

                                            Ouvrage Éditeur (xe fois) Notes  
0  Nicolo-Peccavi ou l'Affaire Dreyfus à Carpentras     Gallimard (1)   NaN  
1                                           Maïtena            Fayard   NaN  
Table 2 - shape (12, 2)
Columns: ['v\xa0· mLauréats du prix Renaudot', 'v\xa0· mLauréats du prix Renaudot.1']
  v · mLauréats du prix Renaudot  \
0                  Prix Renaudot   
1                    Années 1920   

                    v · mLauré

/tmp/ipykernel_3924/879109680.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_renaudot = pd.read_html(response.text)


In [14]:
df_renaudot = tables_renaudot[1].copy()
df_renaudot = df_renaudot.drop(columns=['Unnamed: 1'])
df_renaudot = df_renaudot.rename(columns={
    'Année': 'year',
    'Auteur': 'author',
    'Ouvrage': 'book_title',
    'Éditeur (xe fois)': 'publisher_raw',
    'Notes': 'notes'
})

df_renaudot['publisher'] = df_renaudot['publisher_raw'].str.replace(r'\s*\(\d+\)\s*$', '', regex=True).str.strip()
df_renaudot['publisher_cumulative_count'] = df_renaudot['publisher_raw'].str.extract(r'\((\d+)\)$').astype('Int64')
df_renaudot = df_renaudot.drop(columns=['publisher_raw'])

df_renaudot['prize'] = 'Renaudot'
df_renaudot['country'] = 'France'

df_renaudot = df_renaudot[['prize', 'country', 'year', 'author', 'book_title', 'publisher', 'publisher_cumulative_count', 'notes']]

print(df_renaudot.head())
print(df_renaudot.tail())
print(df_renaudot.info())

      prize country  year             author  \
0  Renaudot  France  1926       Armand Lunel   
1  Renaudot  France  1927    Bernard Nabonne   
2  Renaudot  France  1928         André Obey   
3  Renaudot  France  1929        Marcel Aymé   
4  Renaudot  France  1930  Germaine Beaumont   

                                         book_title  publisher  \
0  Nicolo-Peccavi ou l'Affaire Dreyfus à Carpentras  Gallimard   
1                                           Maïtena     Fayard   
2                             Le Joueur de triangle    Grasset   
3                               La Table-aux-crevés  Gallimard   
4                                             Piège    Lemerre   

   publisher_cumulative_count                             notes  
0                           1                               NaN  
1                        <NA>                               NaN  
2                           1                               NaN  
3                           2                     

In [15]:
df_renaudot.to_csv('renaudot_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_renaudot)} rows")

Saved 100 rows


In [69]:
import re

df_renaudot = pd.read_csv('renaudot_raw.csv')
df_renaudot['author'] = df_renaudot['author'].apply(
    lambda x: re.sub(r'\s*\[\d+\]\s*$', '', str(x)).strip() if pd.notna(x) else x
)
df_renaudot.to_csv('renaudot_raw.csv', index=False, encoding='utf-8')

In [17]:
import requests

url_booker = "https://en.wikipedia.org/wiki/Booker_Prize#Winners"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    response = requests.get(url_booker, headers=headers)
    response.raise_for_status()
    tables_booker = pd.read_html(response.text)
    print(f"Tables found: {len(tables_booker)}")

    for i, t in enumerate(tables_booker[:10]):
        print(f"Table {i} - shape {t.shape}")
        print(f"Columns: {list(t.columns)}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")


Tables found: 4
Table 0 - shape (8, 2)
Columns: ['The Booker Prize', 'The Booker Prize.1']
Table 1 - shape (60, 5)
Columns: ['Year', 'Author', 'Title', 'Genre(s)', 'Country']
Table 2 - shape (7, 2)
Columns: ['vteRecipients of the Booker Prize', 'vteRecipients of the Booker Prize.1']
Table 3 - shape (3, 2)
Columns: ['Authority control databases', 'Authority control databases.1']


/tmp/ipykernel_3924/4292052563.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_booker = pd.read_html(response.text)


In [24]:
df_booker = tables_booker[1].copy()
df_booker = df_booker.rename(columns={
    'Year': 'year',
    'Author': 'author',
    'Title': 'book_title',
    'Country': 'author_country'
})

df_booker['author'] = df_booker['author'].str.replace(r'\[\d+\]', '', regex=True).str.strip()
df_booker['prize'] = 'Booker'
df_booker['country'] = 'UK'

df_booker = df_booker[['prize', 'country', 'year', 'author', 'author_country', 'book_title']]

print(df_booker.head())
print(df_booker.tail())
print(df_booker.info())

    prize country  year          author author_country  \
0  Booker      UK  1969     P. H. Newby            ENG   
1  Booker      UK  1970  Bernice Rubens            WAL   
2  Booker      UK  1971   V. S. Naipaul        UK  TTO   
3  Booker      UK  1972     John Berger            ENG   
4  Booker      UK  1973   J. G. Farrell       ENG  IRL   

                book_title  
0  Something to Answer For  
1       The Elected Member  
2          In a Free State  
3                       G.  
4  The Siege of Krishnapur  
     prize country  year               author author_country  \
55  Booker      UK  2021         Damon Galgut            RSA   
56  Booker      UK  2022  Shehan Karunatilaka            SRI   
57  Booker      UK  2023           Paul Lynch            IRL   
58  Booker      UK  2024      Samantha Harvey            ENG   
59  Booker      UK  2025         David Szalay       CAN  HUN   

                          book_title  
55                       The Promise  
56  The Seven 

In [25]:
df_booker.to_csv('booker_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_booker)} rows")

Saved 60 rows


In [27]:
import requests
from bs4 import BeautifulSoup

url = "https://fr.wikipedia.org/wiki/Prix_Pulitzer_de_la_fiction"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    content = soup.find('div', {'class': 'mw-parser-output'})

    if content:
        items = content.find_all('li')
        for item in items[:20]:
            print(item.get_text())
    else:
        print("Error: 'mw-parser-output' div not found in the page. Check URL or page structure.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


1948 : Pacifique Sud (roman) (Tales of South Pacific) de James A. Michener
1949 : Guard of Honor de James Gould Cozzens
1950 : Orégon-Express (The Way West) de A. B. Guthrie
1951 : La Ville (The Town) de Conrad Richter
1952 : Ouragan sur le Caine (The Caine Mutiny) d'Herman Wouk
1953 : Le Vieil Homme et la Mer (The Old Man and the Sea) d'Ernest Hemingway
1954 : non attribué
1955 : Parabole (A Fable) de William Faulkner
1956 : Andersonville (Andersonville) de MacKinlay Kantor
1957 : non attribué
1958 : Une mort dans la famille (A Death in the Family) de James Agee
1959 : Le Long Voyage de Jaimie Mac Pheeters (The Travels of Jaimie McPheeters) de Robert Lewis Taylor
1960 : Titans (Advise and Consent) d'Allen Drury
1961 : Ne tirez pas sur l'oiseau moqueur (To Kill a Mockingbird) d'Harper Lee
1962 : L'Instant de vérité (The Edge of Sadness) d'Edwin O'Connor
1963 : Les Larrons (The Reivers) de William Faulkner
1964 : non attribué
1965 : Les Gardiens de la maison (The Keepers of the House) d

In [31]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://fr.wikipedia.org/wiki/Prix_Pulitzer_de_la_fiction"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    content = soup.find('div', {'class': 'mw-parser-output'})

    if content:
        items = content.find_all('li')

        records = []
        pattern = re.compile(r'^(\d{4})\s*:\s*(.+)$')

        for item in items:
            text = item.get_text(separator=' ', strip=True)
            text = re.sub(r'\s+', ' ', text)

            match = pattern.match(text)
            if not match:
                continue

            year = int(match.group(1))
            rest = match.group(2).strip()

            if year < 1948 or year > 2025:
                continue

            if 'non attribué' in rest.lower():
                records.append({
                    'year': year,
                    'author': None,
                    'book_title': None,
                    'notes': 'No prize awarded'
                })
                continue

            author_match = re.search(r"\s+d['e]\s+([^()]+?)$", rest)
            if author_match:
                author = author_match.group(1).strip()
                title_part = rest[:author_match.start()].strip()
            else:
                author = None
                title_part = rest

            parentheses = re.findall(r'\(([^)]+)\)', title_part)
            if parentheses:
                book_title = parentheses[-1].strip()
            else:
                book_title = re.sub(r'\s*\([^)]*\)\s*', '', title_part).strip()

            records.append({
                'year': year,
                'author': author,
                'book_title': book_title,
                'notes': None
            })

        df_pulitzer = pd.DataFrame(records)
        df_pulitzer['prize'] = 'Pulitzer Fiction'
        df_pulitzer['country'] = 'USA'
        df_pulitzer['author_country'] = 'USA'
        df_pulitzer['publisher'] = None

        df_pulitzer = df_pulitzer[['prize', 'country', 'year', 'author', 'author_country',
                                    'book_title', 'publisher', 'notes']]

        print(df_pulitzer.head(15))
        print(df_pulitzer.tail(10))
        print(df_pulitzer.shape)
    else:
        print("Error: 'mw-parser-output' div not found in the page. Check URL or page structure.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

               prize country  year               author author_country  \
0   Pulitzer Fiction     USA  1948    James A. Michener            USA   
1   Pulitzer Fiction     USA  1949  James Gould Cozzens            USA   
2   Pulitzer Fiction     USA  1950        A. B. Guthrie            USA   
3   Pulitzer Fiction     USA  1951       Conrad Richter            USA   
4   Pulitzer Fiction     USA  1952          Herman Wouk            USA   
5   Pulitzer Fiction     USA  1953     Ernest Hemingway            USA   
6   Pulitzer Fiction     USA  1954                 None            USA   
7   Pulitzer Fiction     USA  1955     William Faulkner            USA   
8   Pulitzer Fiction     USA  1956     MacKinlay Kantor            USA   
9   Pulitzer Fiction     USA  1957                 None            USA   
10  Pulitzer Fiction     USA  1958           James Agee            USA   
11  Pulitzer Fiction     USA  1959  Robert Lewis Taylor            USA   
12  Pulitzer Fiction     USA  1960    

In [32]:
df_pulitzer.to_csv('pulitzer_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_pulitzer)} rows")

Saved 78 rows


In [70]:
df_pulitzer = pd.read_csv('pulitzer_raw.csv')

df_pulitzer = df_pulitzer[df_pulitzer['year'] != 2023].copy()

new_rows = pd.DataFrame([
    {'prize': 'Pulitzer Fiction', 'country': 'USA', 'year': 2023,
     'author': 'Barbara Kingsolver', 'author_country': 'USA',
     'book_title': 'Demon Copperhead', 'publisher': None, 'notes': 'Co-laureate'},
    {'prize': 'Pulitzer Fiction', 'country': 'USA', 'year': 2023,
     'author': 'Hernan Diaz', 'author_country': 'USA',
     'book_title': 'Trust', 'publisher': None, 'notes': 'Co-laureate'},
])

df_pulitzer = pd.concat([df_pulitzer, new_rows], ignore_index=True)
df_pulitzer = df_pulitzer.sort_values('year').reset_index(drop=True)
df_pulitzer.to_csv('pulitzer_raw.csv', index=False, encoding='utf-8')

In [36]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = "https://fr.wikipedia.org/wiki/Prix_Nobel_de_litt%C3%A9rature"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

for sup in soup.find_all('sup'):
    sup.decompose()

content = soup.find('div', {'class': 'mw-parser-output'})
items = content.find_all('li')

CONNECTORS = {
    'et', ',', 'écrit en', ', écrit en', 'écrit en provençal',
    'provençal', 'alors sous administration russe', '&',
    '(alors sous administration russe)'
}

def is_garbage(part):
    p = part.strip().lower()
    if not p:
        return True
    if p in CONNECTORS:
        return True
    if re.match(r'^[\[\]\(\),\.\s]+$', p):
        return True
    if re.match(r'^\d+$', p) and len(p) <= 3:
        return True
    if p.startswith(','):
        return True
    if 'écrit en' in p:
        return True
    if 'alors sous' in p:
        return True
    return False

records = []

for item in items:
    text = item.get_text(separator='|', strip=True)
    raw_parts = [p.strip() for p in text.split('|')]
    parts = [p for p in raw_parts if not is_garbage(p) and p != ':']

    if not parts:
        continue

    year_match = re.match(r'^(\d{4})', parts[0])
    if not year_match:
        continue

    year = int(year_match.group(1))
    if year < 1901 or year > 2025:
        continue

    content_parts = parts[1:]

    full_text = ' '.join(content_parts).lower()
    if 'non attribué' in full_text or 'non décerné' in full_text or 'pas décerné' in full_text:
        records.append({
            'year': year,
            'author': None,
            'author_country': None,
            'notes': 'No prize awarded'
        })
        continue

    if len(content_parts) >= 2:
        records.append({
            'year': year,
            'author': content_parts[0],
            'author_country': content_parts[1],
            'notes': None
        })

        if len(content_parts) >= 4:
            records.append({
                'year': year,
                'author': content_parts[2],
                'author_country': content_parts[3],
                'notes': 'Co-laureate'
            })

df_nobel = pd.DataFrame(records)
df_nobel['prize'] = 'Nobel Literature'
df_nobel['country'] = 'Sweden'
df_nobel['book_title'] = None
df_nobel['publisher'] = None

df_nobel = df_nobel[['prize', 'country', 'year', 'author', 'author_country',
                      'book_title', 'publisher', 'notes']]

print(df_nobel.head(15))
print(df_nobel.tail(15))
print(df_nobel.shape)

               prize country  year                   author  \
0   Nobel Literature  Sweden  1901          Sully Prudhomme   
1   Nobel Literature  Sweden  1902          Theodor Mommsen   
2   Nobel Literature  Sweden  1903    Bjørnstjerne Bjørnson   
3   Nobel Literature  Sweden  1904         Frédéric Mistral   
4   Nobel Literature  Sweden  1904        José de Echegaray   
5   Nobel Literature  Sweden  1905       Henryk Sienkiewicz   
6   Nobel Literature  Sweden  1906          Giosuè Carducci   
7   Nobel Literature  Sweden  1907          Rudyard Kipling   
8   Nobel Literature  Sweden  1908  Rudolf Christoph Eucken   
9   Nobel Literature  Sweden  1909           Selma Lagerlöf   
10  Nobel Literature  Sweden  1910               Paul Heyse   
11  Nobel Literature  Sweden  1911      Maurice Maeterlinck   
12  Nobel Literature  Sweden  1912        Gerhart Hauptmann   
13  Nobel Literature  Sweden  1913      Rabindranath Tagore   
14  Nobel Literature  Sweden  1914                     

In [39]:
print(df_nobel[df_nobel['year'].isin([1917, 1966, 1974])])

               prize country  year                 author author_country  \
17  Nobel Literature  Sweden  1917  Karl Adolph Gjellerup       Danemark   
18  Nobel Literature  Sweden  1917     Henrik Pontoppidan       Danemark   
64  Nobel Literature  Sweden  1966    Samuel Joseph Agnon         Israël   
65  Nobel Literature  Sweden  1966            Nelly Sachs          Suède   
73  Nobel Literature  Sweden  1974         Eyvind Johnson          Suède   
74  Nobel Literature  Sweden  1974        Harry Martinson          Suède   

   book_title publisher        notes  
17       None      None         None  
18       None      None  Co-laureate  
64       None      None         None  
65       None      None  Co-laureate  
73       None      None         None  
74       None      None  Co-laureate  


In [40]:
mask_1966_bad = (df_nobel['year'] == 1966) & (df_nobel['author'] == 'hébreu')
df_nobel.loc[mask_1966_bad, 'author'] = 'Nelly Sachs'
df_nobel.loc[mask_1966_bad, 'author_country'] = 'Suède'
df_nobel.loc[mask_1966_bad, 'notes'] = 'Co-laureate'

mask_1974_bad = (df_nobel['year'] == 1974) & (df_nobel['author'] == 'Eyvind Johnson')
df_nobel.loc[mask_1974_bad, 'author_country'] = 'Suède'

new_row_1974 = pd.DataFrame([{
    'prize': 'Nobel Literature',
    'country': 'Sweden',
    'year': 1974,
    'author': 'Harry Martinson',
    'author_country': 'Suède',
    'book_title': None,
    'publisher': None,
    'notes': 'Co-laureate'
}])

df_nobel = pd.concat([df_nobel, new_row_1974], ignore_index=True)
df_nobel = df_nobel.sort_values(['year', 'notes'], na_position='first').reset_index(drop=True)

print(df_nobel[df_nobel['year'].isin([1917, 1966, 1974])])

               prize country  year                 author author_country  \
17  Nobel Literature  Sweden  1917  Karl Adolph Gjellerup       Danemark   
18  Nobel Literature  Sweden  1917     Henrik Pontoppidan       Danemark   
64  Nobel Literature  Sweden  1966    Samuel Joseph Agnon         Israël   
65  Nobel Literature  Sweden  1966            Nelly Sachs          Suède   
73  Nobel Literature  Sweden  1974         Eyvind Johnson          Suède   
74  Nobel Literature  Sweden  1974        Harry Martinson          Suède   
75  Nobel Literature  Sweden  1974        Harry Martinson          Suède   

   book_title publisher        notes  
17       None      None         None  
18       None      None  Co-laureate  
64       None      None         None  
65       None      None  Co-laureate  
73       None      None         None  
74       None      None  Co-laureate  
75       None      None  Co-laureate  


In [41]:
df_nobel.to_csv('nobel_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_nobel)} rows")
print(df_nobel.shape)

Saved 132 rows
(132, 8)


In [42]:
print(len(df_nobel))
print(df_nobel.shape)

132
(132, 8)


In [43]:
print(df_nobel[df_nobel.duplicated(subset=['year', 'author'], keep=False)].sort_values('year'))

               prize country  year           author author_country book_title  \
74  Nobel Literature  Sweden  1974  Harry Martinson          Suède       None   
75  Nobel Literature  Sweden  1974  Harry Martinson          Suède       None   

   publisher        notes  
74      None  Co-laureate  
75      None  Co-laureate  


In [44]:
df_nobel = df_nobel.drop_duplicates(subset=['year', 'author'], keep='first').reset_index(drop=True)
print(df_nobel.shape)
print(df_nobel[df_nobel['year'] == 1974])

(131, 8)
               prize country  year           author author_country book_title  \
73  Nobel Literature  Sweden  1974   Eyvind Johnson          Suède       None   
74  Nobel Literature  Sweden  1974  Harry Martinson          Suède       None   

   publisher        notes  
73      None         None  
74      None  Co-laureate  


In [45]:
df_nobel.to_csv('nobel_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_nobel)} rows")
print(df_nobel.shape)

Saved 131 rows
(131, 8)


In [47]:
import requests

url_akutagawa = "https://en.wikipedia.org/wiki/Akutagawa_Prize#Winners"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url_akutagawa, headers=headers)
    response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
    tables_akutagawa = pd.read_html(response.text)
    print(f"Tables found: {len(tables_akutagawa)}")

    for i, t in enumerate(tables_akutagawa[:5]):
        print(f"Table {i} - shape {t.shape}")
        print(f"Columns: {list(t.columns)}")
        print(t.head(15))
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")

Tables found: 7
Table 0 - shape (7, 2)
Columns: ['Akutagawa Prize', 'Akutagawa Prize.1']
                    Akutagawa Prize  \
0  芥川龍之介賞 (Akutagawa Ryūnosuke Shō)   
1                       Awarded for   
2                           Country   
3                      Presented by   
4                            Reward   
5                       First award   
6                           Website   

                                  Akutagawa Prize.1  
0                  芥川龍之介賞 (Akutagawa Ryūnosuke Shō)  
1  Best published literary story by a rising author  
2                                             Japan  
3  Society for the Promotion of Japanese Literature  
4                          ¥1,000,000, pocket watch  
5                                1935; 91 years ago  
6     www.bunshun.co.jp/shinkoukai/award/akutagawa/  
Table 1 - shape (2, 2)
Columns: [0, 1]
   0                                             1
0  上   Indicates the first half of the given year.
1  下  Indicates the secon

/tmp/ipykernel_3924/2043007164.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_akutagawa = pd.read_html(response.text)


In [52]:
import pandas as pd
import re
import requests

url_akutagawa = "https://en.wikipedia.org/wiki/Akutagawa_Prize"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url_akutagawa, headers=headers)
    response.raise_for_status()
    tables_akutagawa = pd.read_html(response.text)

    df_akutagawa = tables_akutagawa[2].copy()
    df_akutagawa = df_akutagawa.drop(columns=['#'])

    def parse_year_session(year_str):
        if pd.isna(year_str):
            return None, None
        match = re.match(r'^(\d{4})\s*([上下]?)', str(year_str))
        if not match:
            return None, None
        year = int(match.group(1))
        session_char = match.group(2)
        if session_char == '上':
            session = 'H1'
        elif session_char == '下':
            session = 'H2'
        else:
            session = None
        return year, session

    df_akutagawa[['year', 'session']] = df_akutagawa['Year'].apply(
        lambda x: pd.Series(parse_year_session(x))
    )

    no_prize_mask = df_akutagawa['Author'].str.contains('No prize awarded', case=False, na=False)

    def clean_author(name):
        if pd.isna(name):
            return None
        name = re.sub(r'\s*\[ja\]\s*', '', str(name))
        name = re.sub(r'\s+refused by the author', '', name)
        return name.strip()

    df_akutagawa['author'] = df_akutagawa['Author'].apply(clean_author)
    df_akutagawa['book_title'] = df_akutagawa['Work']
    df_akutagawa['publisher'] = df_akutagawa['Published in']

    notes_list = []
    for idx, row in df_akutagawa.iterrows():
        if no_prize_mask[idx]:
            notes_list.append('No prize awarded')
        elif 'refused by the author' in str(row['Author']).lower():
            notes_list.append('Refused by the author')
        else:
            notes_list.append(None)
    df_akutagawa['notes'] = notes_list

    df_akutagawa.loc[no_prize_mask, ['author', 'book_title', 'publisher']] = None

    df_akutagawa['prize'] = 'Akutagawa'
    df_akutagawa['country'] = 'Japan'
    df_akutagawa['author_country'] = 'Japan'

    df_akutagawa = df_akutagawa[['prize', 'country', 'year', 'session', 'author', 'author_country',
                                  'book_title', 'publisher', 'notes']]

    print(df_akutagawa.head(20))
    print(df_akutagawa.tail(10))
    print(df_akutagawa.shape)
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")

/tmp/ipykernel_3924/383921268.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_akutagawa = pd.read_html(response.text)


        prize country  year session                            author  \
0   Akutagawa   Japan  1935      H1                  Tatsuzō Ishikawa   
1   Akutagawa   Japan  1935      H2                              None   
2   Akutagawa   Japan  1936      H1                         Takeo Oda   
3   Akutagawa   Japan  1936      H1                    Tomoya Tsuruta   
4   Akutagawa   Japan  1936      H2                      Jun Ishikawa   
5   Akutagawa   Japan  1936      H2                      Uio Tomisawa   
6   Akutagawa   Japan  1937      H1                       Kazuo Ozaki   
7   Akutagawa   Japan  1937      H2                      Ashihei Hino   
8   Akutagawa   Japan  1938      H1                    Gishū Nakayama   
9   Akutagawa   Japan  1938      H2                  Tsuneko Nakazato   
10  Akutagawa   Japan  1939      H1                   Yoshiyuki Handa   
11  Akutagawa   Japan  1939      H1                          Ken Hase   
12  Akutagawa   Japan  1939      H2                

In [53]:
def fix_author(name):
    if pd.isna(name):
        return None
    name = re.sub(r'refused by the author', '', str(name), flags=re.IGNORECASE)
    name = re.sub(r'\s*\(author\)\s*', '', name)
    name = re.sub(r'\s*\[\d+\]\s*', '', name)
    return name.strip()

def fix_text_field(text):
    if pd.isna(text):
        return None
    text = re.sub(r'\s*\[\d+\]\s*', '', str(text))
    return text.strip()

df_akutagawa['author'] = df_akutagawa['author'].apply(fix_author)
df_akutagawa['book_title'] = df_akutagawa['book_title'].apply(fix_text_field)
df_akutagawa['publisher'] = df_akutagawa['publisher'].apply(fix_text_field)

refused_mask = tables_akutagawa[2]['Author'].astype(str).str.contains('refused', case=False, na=False)
df_akutagawa.loc[refused_mask.values, 'notes'] = 'Refused by the author'

print(df_akutagawa[df_akutagawa['notes'].notna()].head(10))
print(df_akutagawa.tail(10))

        prize country  year session       author author_country  \
1   Akutagawa   Japan  1935      H2         None          Japan   
13  Akutagawa   Japan  1940      H1  Takagi Taku          Japan   
17  Akutagawa   Japan  1942      H1         None          Japan   
28  Akutagawa   Japan  1950      H2         None          Japan   
32  Akutagawa   Japan  1952      H1         None          Japan   
37  Akutagawa   Japan  1953      H2         None          Japan   
44  Akutagawa   Japan  1956      H2         None          Japan   
48  Akutagawa   Japan  1958      H2         None          Japan   
50  Akutagawa   Japan  1959      H2         None          Japan   
53  Akutagawa   Japan  1961      H1         None          Japan   

                    book_title publisher                  notes  
1                         None      None       No prize awarded  
13  Uta to mon no tate (歌と門の盾)      None  Refused by the author  
17                        None      None       No prize awarded 

In [54]:
df_akutagawa.to_csv('akutagawa_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_akutagawa)} rows")

Saved 222 rows


In [56]:
import requests
import pandas as pd

url_naoki = "https://en.wikipedia.org/wiki/Naoki_Prize"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url_naoki, headers=headers)
    response.raise_for_status()
    tables_naoki = pd.read_html(response.text)
    print(f"Tables found: {len(tables_naoki)}")

    for i, t in enumerate(tables_naoki[:6]):
        print(f"Table {i} - shape {t.shape}")
        print(f"Columns: {list(t.columns)}")
        print(t.head(3))
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")

Tables found: 5
Table 0 - shape (6, 2)
Columns: ['Naoki Prize', 'Naoki Prize.1']
                  Naoki Prize  \
0  直木三十五賞 (Naoki Sanjūgo Shō)   
1                 Awarded for   
2                     Country   

                                      Naoki Prize.1  
0                        直木三十五賞 (Naoki Sanjūgo Shō)  
1  Best work of popular literature by rising author  
2                                             Japan  
Table 1 - shape (100, 4)
Columns: [0, 1, 2, 3]
      0  1                    2  \
0  1935  1  Matsutarō Kawaguchi   
1  1935  2           Uko Washio   
2  1936  3      Chōgorō Kaionji   

                                                   3  
0  Tsuruhachi Tsurujirō (鶴八鶴次郎), Fūryū Fukagawa U...  
1  Yoshinochō Taiheiki (吉野朝太平記; lit. Chronicles o...  
2  Tenshō Onna Gassen (天正女合戦; lit. Tenshō Women's...  
Table 2 - shape (74, 4)
Columns: [0, 1, 2, 3]
      0    1                              2  \
0  1989  101  Nejime Shōichi Akira Sasakura   
1  1989  102       Sei

/tmp/ipykernel_3924/189969545.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_naoki = pd.read_html(response.text)


In [58]:
import pandas as pd
import re
import requests

url_naoki = "https://en.wikipedia.org/wiki/Naoki_Prize"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(url_naoki, headers=headers)
    response.raise_for_status()
    tables_naoki = pd.read_html(response.text)

    df_part1 = tables_naoki[1].copy()
    df_part2 = tables_naoki[2].copy()

    df_naoki_raw = pd.concat([df_part1, df_part2], ignore_index=True)
    df_naoki_raw.columns = ['year', 'prize_number', 'author_raw', 'work_raw']

    print(df_naoki_raw.head(10))
    print(df_naoki_raw.tail(10))
    print(f"Total rows: {len(df_naoki_raw)}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching URL: {e}")
except ValueError as e:
    print(f"Error parsing HTML: {e}")

   year prize_number           author_raw  \
0  1935            1  Matsutarō Kawaguchi   
1  1935            2           Uko Washio   
2  1936            3      Chōgorō Kaionji   
3  1936            4        Takatarō Kigi   
4  1937            5     No prize awarded   
5  1937            6         Masuji Ibuse   
6  1938            7      Sotoo Tachibana   
7  1938            8          Tadao Ooike   
8  1939            9     No prize awarded   
9  1939           10     No prize awarded   

                                            work_raw  
0  Tsuruhachi Tsurujirō (鶴八鶴次郎), Fūryū Fukagawa U...  
1  Yoshinochō Taiheiki (吉野朝太平記; lit. Chronicles o...  
2  Tenshō Onna Gassen (天正女合戦; lit. Tenshō Women's...  
3          Jinsei no Ahō (人生の阿呆; lit. A Fool's Life)  
4                                                NaN  
5  John Manjirō Hyōryūki (ジョン萬次郎漂流記; lit. The Dri...  
6  Narin-denka e no Kaisō (ナリン殿下への回想; lit. Recoll...  
7  Kabuto (兜首; lit. Helmet), Akitaguchi no Kyōdai...  
8        

/tmp/ipykernel_3924/2064419696.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_naoki = pd.read_html(response.text)


In [60]:
df_naoki = df_naoki_raw.copy()

df_naoki['year'] = pd.to_numeric(df_naoki['year'], errors='coerce').astype('Int64')

def assign_session(prize_number):
    if pd.isna(prize_number):
        return None
    cleaned_prize_number = re.sub(r'\[\d+\]', '', str(prize_number)).strip()
    if not cleaned_prize_number:
        return None
    return 'H1' if int(cleaned_prize_number) % 2 == 1 else 'H2'

df_naoki['session'] = df_naoki['prize_number'].apply(assign_session)

no_prize_mask = df_naoki['author_raw'].astype(str).str.contains('No prize awarded|No award', case=False, na=False)

def clean_author(name):
    if pd.isna(name):
        return None
    name = re.sub(r'\s*\[ja\]\s*', '', str(name))
    name = re.sub(r'refused by the author', '', name, flags=re.IGNORECASE)
    name = re.sub(r'\s*\(author\)\s*', '', name)
    name = re.sub(r'\s*\[\d+\]\s*', '', name)
    return name.strip()

def clean_work(work):
    if pd.isna(work):
        return None
    work = re.sub(r'\s*\[\d+\]\s*', '', str(work))
    return work.strip()

df_naoki['author'] = df_naoki['author_raw'].apply(clean_author)
df_naoki['book_title'] = df_naoki['work_raw'].apply(clean_work)

df_naoki['notes'] = None
df_naoki.loc[no_prize_mask, 'notes'] = 'No prize awarded'
df_naoki.loc[no_prize_mask, ['author', 'book_title']] = None

refused_mask = df_naoki['author_raw'].astype(str).str.contains('refused', case=False, na=False)
df_naoki.loc[refused_mask, 'notes'] = 'Refused by the author'

df_naoki['prize'] = 'Naoki'
df_naoki['country'] = 'Japan'
df_naoki['author_country'] = 'Japan'
df_naoki['publisher'] = None

df_naoki = df_naoki[['prize', 'country', 'year', 'session', 'author', 'author_country',
                      'book_title', 'publisher', 'notes']]

print(df_naoki.head(20))
print(df_naoki.tail(10))
print(f"Total: {len(df_naoki)}")
print(f"With laureate: {df_naoki['author'].notna().sum()}")

    prize country  year session                          author  \
0   Naoki   Japan  1935      H1             Matsutarō Kawaguchi   
1   Naoki   Japan  1935      H2                      Uko Washio   
2   Naoki   Japan  1936      H1                 Chōgorō Kaionji   
3   Naoki   Japan  1936      H2                   Takatarō Kigi   
4   Naoki   Japan  1937      H1                            None   
5   Naoki   Japan  1937      H2                    Masuji Ibuse   
6   Naoki   Japan  1938      H1                 Sotoo Tachibana   
7   Naoki   Japan  1938      H2                     Tadao Ooike   
8   Naoki   Japan  1939      H1                            None   
9   Naoki   Japan  1939      H2                            None   
10  Naoki   Japan  1940      H1  Chiyo Tsutsumi Sensuke Kawachi   
11  Naoki   Japan  1940      H2                  Genzō Murakami   
12  Naoki   Japan  1941      H1                     Sōjū Kimura   
13  Naoki   Japan  1941      H2                            Non

In [62]:
import pandas as pd

suspects = df_naoki[df_naoki['author'].notna()].copy()
suspects['word_count'] = suspects['author'].str.split().str.len()
co_laureate_candidates = suspects[suspects['word_count'] >= 4]
print(co_laureate_candidates[['year', 'session', 'author']])
print(f"Total candidates: {len(co_laureate_candidates)}")

     year session                                author
10   1940      H1        Chiyo Tsutsumi Sensuke Kawachi
15   1942      H2             Norio Taoka Takio Kanzaki
22   1950      H1               Hidemi Kon Itoko Koyama
25   1951      H2         Juran Hisao Renzaburō Shibata
31   1954      H2            Haruo Umezaki Yukio Togawa
33   1955      H2                  Jirō Nitta Eikan Kyū
34   1956      H1               Norio Nanjō Kanichi Kon
35   1956      H2                Tōkō Kon Miharu Hozumi
38   1958      H1           Toyoko Yamasaki Eiji Shinba
39   1958      H2         Saburō Shiroyama Kyō Takigawa
40   1959      H1          Kieko Watanabe Yumie Hiraiwa
41   1959      H2            Ryōtarō Shiba Yasuji Toita
43   1960      H2        Daikichi Terauchi Jūgo Kuroiwa
47   1962      H2      Hitomi Yamaguchi Sonoko Sugimoto
49   1963      H2               Tsuruo Andō Yoshie Wada
51   1964      H2            Michiko Nagai Atsuko Anzai
53   1965      H2         Yūkichi Shinbashi Jihe

In [63]:
import pandas as pd

co_laureate_fixes = {
    (1940, 'H1'): ['Chiyo Tsutsumi', 'Sensuke Kawachi'],
    (1942, 'H2'): ['Norio Taoka', 'Takio Kanzaki'],
    (1950, 'H1'): ['Hidemi Kon', 'Itoko Koyama'],
    (1951, 'H2'): ['Juran Hisao', 'Renzaburō Shibata'],
    (1954, 'H2'): ['Haruo Umezaki', 'Yukio Togawa'],
    (1955, 'H2'): ['Jirō Nitta', 'Eikan Kyū'],
    (1956, 'H1'): ['Norio Nanjō', 'Kanichi Kon'],
    (1956, 'H2'): ['Tōkō Kon', 'Miharu Hozumi'],
    (1958, 'H1'): ['Toyoko Yamasaki', 'Eiji Shinba'],
    (1958, 'H2'): ['Saburō Shiroyama', 'Kyō Takigawa'],
    (1959, 'H1'): ['Kieko Watanabe', 'Yumie Hiraiwa'],
    (1959, 'H2'): ['Ryōtarō Shiba', 'Yasuji Toita'],
    (1960, 'H2'): ['Daikichi Terauchi', 'Jūgo Kuroiwa'],
    (1962, 'H2'): ['Hitomi Yamaguchi', 'Sonoko Sugimoto'],
    (1963, 'H2'): ['Tsuruo Andō', 'Yoshie Wada'],
    (1964, 'H2'): ['Michiko Nagai', 'Atsuko Anzai'],
    (1965, 'H2'): ['Yūkichi Shinbashi', 'Jihei Chiba'],
    (1968, 'H2'): ['Shunshin Chin', 'Mitsugu Saotome'],
    (1972, 'H1'): ['Hisashi Inoue', 'Kenjō Tsunabuchi'],
    (1974, 'H2'): ['Ryō Hanmura', 'Magoroku Ide'],
    (1978, 'H1'): ['Yō Tsumoto', 'Takehiro Irokawa'],
    (1978, 'H2'): ['Tomiko Miyao', 'Natsuo Ariake'],
    (1980, 'H1'): ['Kuniko Mukōda', 'Kageki Shimoda'],
    (1981, 'H2'): ['Kôhei Tsuka', 'Akira Mitsuoka'],
    (1982, 'H1'): ['Yūsuke Fukada', 'Tomomi Muramatsu'],
    (1983, 'H2'): ['Takurō Kanki', 'Osamu Takahashi'],
    (1984, 'H1'): ['Mikihiko Renjō', 'Toshizō Nanba'],
    (1985, 'H2'): ['Seigo Morita', 'Mariko Hayashi'],
    (1988, 'H1'): ['Tamio Kageyama', 'Masaaki Nishiki'],
    (1988, 'H2'): ['Akiko Sugimoto', 'Shizuko Tōdō'],
    (1989, 'H1'): ['Nejime Shōichi', 'Akira Sasakura'],
    (1989, 'H2'): ['Seiji Hashikawa', 'Ryō Hara'],
    (1991, 'H1'): ['Masamitsu Miyagitani', 'Sunao Ashihara'],
    (1991, 'H2'): ['Yoshio Takahashi', 'Katsuhiko Takahashi'],
    (1993, 'H1'): ['Kaoru Takamura', 'Aiko Kitahara'],
    (1993, 'H2'): ['Arimasa Osawa', 'Masayoshi Satō'],
    (1994, 'H1'): ['Akihiko Nakamura', 'Yasuhisa Ebisawa'],
    (1995, 'H2'): ['Mariko Koike', 'Iori Fujiwara'],
    (1997, 'H1'): ['Setsuko Shinoda', 'Jirō Asada'],
    (1999, 'H1'): ['Kenichi Satō', 'Natsuo Kirino'],
    (2000, 'H1'): ['Yoichi Funado', 'Kazuki Kaneshiro'],
    (2000, 'H2'): ['Kiyoshi Shigematsu', 'Fumio Yamamoto'],
    (2001, 'H2'): ['Ichiriki Yamamoto', 'Kei Yuikawa'],
    (2003, 'H1'): ['Ira Ishida', 'Yuka Murayama'],
    (2003, 'H2'): ['Kaori Ekuni', 'Natsuhiko Kyogoku'],
    (2004, 'H1'): ['Hideo Okuda', 'Tatsuya Kumagai'],
    (2006, 'H1'): ['Shion Miura', 'Eto Mori'],
    (2008, 'H2'): ['Arata Tendo', "Ken'ichi Yamamoto"],
    (2009, 'H2'): ['Jō Sasaki', 'Kazufumi Shiraishi'],
    (2010, 'H2'): ['Nobori Kiuchi', 'Shūsuke Michio'],
    (2012, 'H2'): ['Asai Ryo', 'Abe Ryutaro'],
    (2013, 'H2'): ['Makate Asai', 'Kaoruko Himeno'],
    (2021, 'H1'): ['Toko Sawada', 'Norikazu Sato'],
    (2021, 'H2'): ['Shogo Imamura', 'Honobu Yonezawa'],
    (2022, 'H2'): ['Satoshi Ogawa', 'Akane Chihaya'],
    (2023, 'H1'): ['Ryōsuke Kakine', 'Sayako Nagai'],
    (2023, 'H2'): ['Akiko Kawasaki', 'Manabu Makime'],
}

new_rows = []
for (year, session), authors in co_laureate_fixes.items():
    mask = (df_naoki['year'] == year) & (df_naoki['session'] == session)
    if not mask.any():
        print(f"Warning: no match for {year} {session}")
        continue

    original_row = df_naoki[mask].iloc[0].copy()
    df_naoki.loc[mask, 'author'] = authors[0]
    df_naoki.loc[mask, 'notes'] = 'Co-laureate'

    new_row = original_row.copy()
    new_row['author'] = authors[1]
    new_row['notes'] = 'Co-laureate'
    new_rows.append(new_row)

if new_rows:
    df_naoki = pd.concat([df_naoki, pd.DataFrame(new_rows)], ignore_index=True)
    df_naoki = df_naoki.sort_values(['year', 'session', 'notes'], na_position='last').reset_index(drop=True)

print(f"Final shape: {df_naoki.shape}")
print(f"Total laureates: {df_naoki['author'].notna().sum()}")
print(f"Co-laureates: {(df_naoki['notes'] == 'Co-laureate').sum()}")

Final shape: (231, 9)
Total laureates: 201
Co-laureates: 114


In [64]:
print(df_naoki[df_naoki['year'].isna()])

     prize country  year session                       author author_country  \
225  Naoki   Japan  <NA>      H1              Hiroko Minagawa          Japan   
226  Naoki   Japan  <NA>      H1  Amy Yamada Ichirō Shiraishi          Japan   
227  Naoki   Japan  <NA>      H1                   Asa Nonami          Japan   
228  Naoki   Japan  <NA>      H2      Gō Ōsaka Shinpei Tokiwa          Japan   
229  Naoki   Japan  <NA>      H2                    Makio Abe          Japan   
230  Naoki   Japan  <NA>      H2                 Masako Bandō          Japan   

                                            book_title publisher notes  
225                Koi Kurenai (恋紅; lit. Crimson Love)      None  None  
226  Sōru Myūjikku Rabāzu Onrī (ソウル・ミュージック・ラバーズ・オンリ...      None  None  
227                    The Hunter (凍える牙, Kogoeru Kiba)      None  None  
228  The Red Star of Cadiz (カディスの赤い星, Kadisu no Aka...      None  None  
229  Sorezore no Tsuigakushō (それぞれの終楽章; lit. The En...      None  None  
2

In [65]:
print("Searches without match:")
for (year, session), authors in co_laureate_fixes.items():
    mask = (df_naoki['year'] == year) & (df_naoki['session'] == session)
    if not mask.any():
        print(f"  {year} {session} : {authors}")

Searches without match:


In [66]:
na_fixes = {
    225: 1986,
    226: 1987,
    227: 1996,
    228: 1986,
    229: 1987,
    230: 1996,
}

for idx, year in na_fixes.items():
    df_naoki.loc[idx, 'year'] = year

print(df_naoki.loc[list(na_fixes.keys())])

     prize country  year session                       author author_country  \
225  Naoki   Japan  1986      H1              Hiroko Minagawa          Japan   
226  Naoki   Japan  1987      H1  Amy Yamada Ichirō Shiraishi          Japan   
227  Naoki   Japan  1996      H1                   Asa Nonami          Japan   
228  Naoki   Japan  1986      H2      Gō Ōsaka Shinpei Tokiwa          Japan   
229  Naoki   Japan  1987      H2                    Makio Abe          Japan   
230  Naoki   Japan  1996      H2                 Masako Bandō          Japan   

                                            book_title publisher notes  
225                Koi Kurenai (恋紅; lit. Crimson Love)      None  None  
226  Sōru Myūjikku Rabāzu Onrī (ソウル・ミュージック・ラバーズ・オンリ...      None  None  
227                    The Hunter (凍える牙, Kogoeru Kiba)      None  None  
228  The Red Star of Cadiz (カディスの赤い星, Kadisu no Aka...      None  None  
229  Sorezore no Tsuigakushō (それぞれの終楽章; lit. The En...      None  None  
2

In [67]:
late_fixes = {
    (1987, 'H1'): ['Amy Yamada', 'Ichirō Shiraishi'],
    (1986, 'H2'): ['Gō Ōsaka', 'Shinpei Tokiwa'],
}

new_rows = []
for (year, session), authors in late_fixes.items():
    mask = (df_naoki['year'] == year) & (df_naoki['session'] == session) & \
           (df_naoki['author'].astype(str).str.contains(authors[0].split()[0], na=False))

    if not mask.any():
        print(f"Warning: no match for {year} {session}")
        continue

    original_row = df_naoki[mask].iloc[0].copy()
    df_naoki.loc[mask, 'author'] = authors[0]
    df_naoki.loc[mask, 'notes'] = 'Co-laureate'

    new_row = original_row.copy()
    new_row['author'] = authors[1]
    new_row['notes'] = 'Co-laureate'
    new_rows.append(new_row)

if new_rows:
    df_naoki = pd.concat([df_naoki, pd.DataFrame(new_rows)], ignore_index=True)

df_naoki = df_naoki.sort_values(['year', 'session', 'notes'], na_position='last').reset_index(drop=True)

print(f"Final shape: {df_naoki.shape}")
print(f"Total laureates: {df_naoki['author'].notna().sum()}")
print(f"Co-laureates: {(df_naoki['notes'] == 'Co-laureate').sum()}")
print()
print("Verification 1986-1987-1996:")
print(df_naoki[df_naoki['year'].isin([1986, 1987, 1996])][['year', 'session', 'author', 'notes']])

Final shape: (233, 9)
Total laureates: 203
Co-laureates: 118

Verification 1986-1987-1996:
     year session            author        notes
122  1986      H1   Hiroko Minagawa         None
123  1986      H2          Gō Ōsaka  Co-laureate
124  1986      H2    Shinpei Tokiwa  Co-laureate
125  1987      H1        Amy Yamada  Co-laureate
126  1987      H1  Ichirō Shiraishi  Co-laureate
127  1987      H2         Makio Abe         None
154  1996      H1        Asa Nonami         None
155  1996      H2      Masako Bandō         None


In [68]:
df_naoki.to_csv('naoki_raw.csv', index=False, encoding='utf-8')
print(f"Saved {len(df_naoki)} rows")

Saved 233 rows
